# Fashion MNIST 기본 오토인코더: TensorFlow에서 PyTorch로
[TensorFlow 원본 튜토리얼](https://www.tensorflow.org/tutorials/generative/autoencoder?hl=ko)의 첫 번째 예제를 옮겼다. 옷 종류를 분류하는 대신 입력 이미지 자체를 정답으로 사용한다.

인코더는 이미지 x를 64개의 숫자 h로 표현하고, 디코더는 h에서 복원 이미지 x′를 만든다. 학습할 때는 x와 x′의 픽셀 차이가 줄어들도록 두 레이어의 가중치를 조정한다.

| TensorFlow / Keras | 이 노트북의 PyTorch 코드 |
|---|---|
| Model, call | nn.Module, forward |
| Dense와 activation | nn.Linear와 별도 활성화 함수 |
| compile, fit | 손실 함수와 옵티마이저를 설정하고 학습·검증 반복문 실행 |

Python 환경에 torch, torchvision, matplotlib가 필요하다. 첫 데이터 다운로드에는 인터넷 연결이 필요하다. 위에서부터 실행한다.

## 1. 실행 컴퓨터와 장치 확인

In [ ]:
import socket
import platform
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("실행 컴퓨터:", socket.gethostname())
print("운영체제:", platform.system(), platform.release())
print("선택한 장치:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(device))

## 2. 데이터 준비
픽셀을 0~1로 바꾸고 채널 축을 제거해 원본 예제처럼 한 장을 (28, 28)로 다룬다. 레이블은 학습에 사용하지 않는다.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda image: image.squeeze(0)),
])
train_dataset = datasets.FashionMNIST("./data", train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)

print(train_dataset.data.shape, test_dataset.data.shape)
image, _ = train_dataset[0]
print(image.shape, image.dtype)

## 3. 모델 정의
이미지 한 장당 784개 픽셀 → 중간 표현 64개 값 → 복원된 784개 픽셀. 배치 차원은 그대로 유지한다.

In [ ]:
latent_dim = 64

class Autoencoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.latent_dim = latent_dim
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, latent_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 784),
            nn.Sigmoid(),
            nn.Unflatten(1, (28, 28)),
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

autoencoder = Autoencoder(latent_dim).to(device)
print("모델 장치:", next(autoencoder.parameters()).device)

## 4. 학습과 검증
원본의 fit(x_train, x_train)에 대응한다. 한 에포크마다 학습 데이터를 섞고 테스트 데이터로 복원 오차를 확인한다. 프레임워크별 기본 초기화와 Adam 설정 차이로 손실 수치까지 같지는 않을 수 있다.

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=0.001)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
for epoch in range(10):
    autoencoder.train()
    train_loss = 0.0
    for images, _ in train_loader:
        images = images.to(device)
        optimizer.zero_grad()
        reconstructed = autoencoder(images)
        loss = criterion(reconstructed, images)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)

    autoencoder.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            reconstructed = autoencoder(images)
            loss = criterion(reconstructed, images)
            val_loss += loss.item() * images.size(0)

    train_loss /= len(train_dataset)
    val_loss /= len(test_dataset)
    print(f"Epoch {epoch + 1}/10 - loss: {train_loss:.4f} - val_loss: {val_loss:.4f}")

## 5. 복원 결과 비교
인코딩과 디코딩은 선택한 장치에서 수행한다. 결과를 NumPy와 Matplotlib에서 사용할 때 CPU로 가져온다.

In [ ]:
x_test = torch.cat([images for images, _ in test_loader], dim=0).to(device)
autoencoder.eval()
with torch.no_grad():
    encoded = autoencoder.encoder(x_test)
    decoded = autoencoder.decoder(encoded)

encoded_imgs = encoded.cpu().numpy()
decoded_imgs = decoded.cpu().numpy()
print(encoded_imgs.shape, decoded_imgs.shape)

In [ ]:
n = 10
fig, axes = plt.subplots(2, n, figsize=(20, 4))
for i in range(n):
    axes[0, i].imshow(x_test[i].cpu().numpy(), cmap="gray")
    axes[0, i].set_title("original")
    axes[0, i].axis("off")
    axes[1, i].imshow(decoded_imgs[i], cmap="gray")
    axes[1, i].set_title("reconstructed")
    axes[1, i].axis("off")
plt.show()